Quick and dirty notebook for simulation of coherent scattering data of magnetic samples in fraunhofer far-field regime

# Import

In [ ]:
# Import general libraries
import sys, os
from os.path import join, split
from importlib import reload
from copy import deepcopy
from tqdm.auto import tqdm

import numpy as np

# scipy
import scipy

# plotting
import matplotlib.pyplot as plt

# Interactive plotting
import ipywidgets
%matplotlib widget
plt.rcParams["figure.constrained_layout.use"] = True

In [ ]:
# Imports from our own codebase
from scattering_calculator.experimental_conditions import detector, light_beam
from scattering_calculator.sample_generator import pattern_generator, structures
from scattering_calculator.interactive.interactive_widgets import cimshow

# Experimental setup

In [ ]:
# Basic parameters for the simulation
sample_shape = (2048, 2048)  # in pixels
real_space_pixel_size = 1e-6  # in m

# ===================
# X-ray Source
# ===================
x_ray_energy = 59.6  # eV
x_ray_photon_flux = 1e10  # Photons per pulse
beam_params = light_beam.beam_parameters(
    x_ray_energy, x_ray_photon_flux
)

# ==================
# Geometry
# ==================
detector_pixel_size = 10e-6  # in m
detector_pixel_shape = (2048, 2048)
detector_distance = 0.15  # in m

# Optional: Define a beamstop
beamstop_radius = 0.5e-3  # in m
beamstop_distance = 0.001  # in m
beamstop_center = np.array(detector_pixel_shape) // 2  # in px

# ==================
# Setup
# ==================

# Basic camera parameters
exp_detector = detector.detector_layout(
    pixel_size=detector_pixel_size,
    shape=detector_pixel_shape,
    distance_sample_detector=detector_distance,
)

# Add beamstop to detector layout
beamstop = detector.beamstop(exp_detector, beamstop_distance)
beamstop.create_circle_beamstop(beamstop_center, beamstop_radius,use_real_space_coordinates=True)
bs_mask = beamstop.return_beamstop()
exp_detector.assign_beamstop(bs_mask)

In [ ]:
# Plot beamstop
extent_det_real = exp_detector.get_detector_extent_real_space()

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(exp_detector.beamstop_mask)
ax[0].set_title("Beamstop in px")
ax[1].imshow(exp_detector.beamstop_mask, extent=1e3*extent_det_real)
ax[1].set_title("Beamstop in mm")
ax[1].set_xlabel("x in mm")
ax[1].set_ylabel("y in mm")

# Illumination function

In [ ]:
reload(light_beam)

In [ ]:
# Params for gaussian beam
illumination_function = "gaussian"
illumination_center = np.array(sample_shape) // 2  # in px
illumination_focus_distance = 0#in m
illumination_fwhm = 30e-6  # in m

illumination = light_beam.illumination(beam_params, sample_shape, real_space_pixel_size)

# Comment: Check gauss_beam function for different focus distances
illumination.gauss_beam(
    illumination_center,
    illumination_focus_distance,
    illumination_fwhm,
)
illumination_wavefield = illumination.return_illumination()
extent_illumination_real = illumination.get_illumination_extent_real_space()
illumination.visualize_illumination()

# Sample

In [ ]:
reload(structures)

## Front Aperture

In [ ]:
front_aperture_radius = 30e-6  # in m
front_aperture = structures.Apertures(sample_shape, real_space_pixel_size)
front_aperture.create_circle_aperture(
    center=(sample_shape[0] // 2, sample_shape[1] // 2),
    radius=front_aperture_radius,
    use_real_space_coordinates=True
)
front_aperture.visualize_aperture()

## Layer structure

In [ ]:
# We should move this into a class as well, values here at 59.5eV, but we should be able to easily change them for different energies. We can also add more materials as needed.
# use some databases
refractive_indices = dict()

refractive_indices["vacuum"] = 1.0 + 0j
refractive_indices["perfect_absorption_mask"] = -1j * 1e6

refractive_indices["SiN"] = 1 - 0.0812632814 - 1j * 0.0325259529
refractive_indices["Ta"] = 1 - 0.14699 - 1j * 0.12070
refractive_indices["Pt"] = 1 - 0.09758 - 1j * 0.18878

refractive_indices["Co"] = 1- 0.00452 - 1j * 0.12677
refractive_indices["Co_xmcd"] = - 0.01474 + 1j * 0.0077 

In [ ]:
material_params = structures.material_params(refractive_indices)

In [ ]:
non_magnetic_structure = structures.Structure(
    name="Test Structure",
    material_params=material_params
)
non_magnetic_structure.add_layer("SiN", thickness=100e-9)
non_magnetic_structure.add_layer("Ta", thickness=3e-9)
for i in range(15):
    non_magnetic_structure.add_layer("Co", thickness=1e-9)
    non_magnetic_structure.add_layer("Pt", thickness=3e-9)                                                      
non_magnetic_structure.add_layer("Co", thickness=1e-9)

non_magnetic_structure.visualize_structure()
non_magnetic_structure.return_total_effective_refractive_index()

In [ ]:
magnetic_structure = structures.Structure(
    name="Test Structure", material_params=material_params
)
magnetic_structure.add_layer("Co_xmcd", thickness=2e-9)
for i in range(15):
    magnetic_structure.add_layer("Co_xmcd", thickness=1e-9)

magnetic_structure.visualize_structure()
magnetic_structure.return_total_effective_refractive_index()

## Magnetic pattern